# Tarea 4: Fractal de Newton en GPU

**Requisito**: `Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4`.

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import time
import numpy as np
import cupy as cp
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

print(cp.cuda.runtime.getDeviceProperties(0)["name"].decode())
print(jax.devices())

Por defecto JAX reserva el 75% de la memoria de la GPU al iniciar. La variable `XLA_PYTHON_CLIENT_PREALLOCATE` lo desactiva para que CuPy y JAX convivan en el mismo *notebook*.

## Código base en NumPy

In [ ]:
MAX_ITER = 100
TOL = 1e-6
ROOTS = np.array([1.0, np.exp(2j * np.pi / 3), np.exp(4j * np.pi / 3)], dtype=np.complex64)

REGIONS = {
    "clasica": (-2.0, 2.0, -2.0, 2.0),
    "frontera_central": (-0.3, 0.3, -0.3, 0.3),
    "frontera_lateral": (0.3, 0.9, 0.3, 0.9),
}


def grid(region, W):
    x_min, x_max, y_min, y_max = REGIONS[region]
    x = np.linspace(x_min, x_max, W, dtype=np.float32)
    y = np.linspace(y_min, y_max, W, dtype=np.float32)
    return (x[None, :] + 1j * y[:, None]).astype(np.complex64)


def newton_numpy(z0):
    z = z0.copy()
    count = np.zeros(z.shape, dtype=np.int32)
    active = np.ones(z.shape, dtype=bool)
    for _ in range(MAX_ITER):
        z_new = z[active] - (z[active] ** 3 - 1) / (3 * z[active] ** 2)
        converged = np.abs(z_new - z[active]) < TOL
        z[active] = z_new
        count[active] += 1
        active[active] = ~converged
        if not active.any():
            break
    root_idx = np.full(z.shape, -1, dtype=np.int32)
    for i, r in enumerate(ROOTS):
        root_idx[np.abs(z - r) < 1e-3] = i
    return root_idx, count


def graficar(root_idx, count, region):
    colors = np.array([[1.0, 0.2, 0.2], [0.2, 1.0, 0.2], [0.2, 0.2, 1.0]])
    shade = 1 - 0.85 * np.clip(count / 30, 0, 1)
    img = np.zeros(root_idx.shape + (3,))
    for i, color in enumerate(colors):
        mask = root_idx == i
        img[mask] = color * shade[mask, None]
    plt.figure(figsize=(6, 6))
    plt.imshow(img, origin="lower", extent=REGIONS[region])
    plt.title(region)
    plt.xlabel("Re(z)")
    plt.ylabel("Im(z)")
    plt.show()

In [ ]:
z0 = grid("clasica", 1024)
root_numpy, count_numpy = newton_numpy(z0)
graficar(root_numpy, count_numpy, "clasica")

## Medición de tiempos

En CuPy se usan eventos CUDA. En JAX se llama a `jax.block_until_ready()`. En ambos casos se descarta la primera llamada (*warm-up*).

In [ ]:
def medir_numpy(z0):
    t = time.perf_counter()
    newton_numpy(z0)
    return time.perf_counter() - t


def medir_cupy(z0):
    newton_cupy(z0)
    inicio = cp.cuda.Event()
    fin = cp.cuda.Event()
    inicio.record()
    newton_cupy(z0)
    fin.record()
    fin.synchronize()
    return cp.cuda.get_elapsed_time(inicio, fin) / 1e3


def medir_rawkernel(z0):
    newton_rawkernel(z0)
    inicio = cp.cuda.Event()
    fin = cp.cuda.Event()
    inicio.record()
    newton_rawkernel(z0)
    fin.record()
    fin.synchronize()
    return cp.cuda.get_elapsed_time(inicio, fin) / 1e3


def medir_jax(z0):
    jax.block_until_ready(newton_jax(z0))
    t = time.perf_counter()
    jax.block_until_ready(newton_jax(z0))
    return time.perf_counter() - t


def mismatch_fraction(a, b):
    return np.mean(np.asarray(a) != np.asarray(b))

## (a) CuPy vectorizado

In [ ]:
def newton_cupy(z0):
    # TODO: misma lógica que newton_numpy, usando arreglos de CuPy.
    # Debe retornar (root_idx, count) como arreglos de CuPy.
    pass

z0_gpu = cp.asarray(z0)
root_cp, count_cp = newton_cupy(z0_gpu)
print("mismatch fraction:", mismatch_fraction(root_numpy, cp.asnumpy(root_cp)))
print(f"numpy: {medir_numpy(z0):.3f} s")
print(f"cupy:  {medir_cupy(z0_gpu):.4f} s")

## (b) CuPy RawKernel

In [ ]:
KERNEL = r'''
extern "C" __global__
void newton(const float* x, const float* y, int* root_idx, int* count,
            int n, int max_iter, float tol)
{
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    if (i >= n) {
        return;
    }
    // TODO: iterar el método de Newton para el pixel i, partiendo de z0 con parte real x[i] y parte imaginaria y[i].
    // Salir del loop en cuanto el pixel converge.
    // Guardar en root_idx[i] la raíz a la que convergió (-1 si no convergió)
    // y en count[i] el número de iteraciones.
}
'''
newton_kernel = cp.RawKernel(KERNEL, "newton")


def newton_rawkernel(z0):
    x = cp.ascontiguousarray(z0.real).ravel()
    y = cp.ascontiguousarray(z0.imag).ravel()
    n = x.size
    root_idx = cp.empty(n, dtype=cp.int32)
    count = cp.empty(n, dtype=cp.int32)
    threads = 256
    blocks = (n + threads - 1) // threads
    newton_kernel((blocks,), (threads,),
                  (x, y, root_idx, count, np.int32(n), np.int32(MAX_ITER), np.float32(TOL)))
    return root_idx.reshape(z0.shape), count.reshape(z0.shape)


root_rk, count_rk = newton_rawkernel(z0_gpu)
print("mismatch fraction:", mismatch_fraction(root_numpy, cp.asnumpy(root_rk)))
print(f"rawkernel: {medir_rawkernel(z0_gpu):.4f} s")

## (c) JAX

In [ ]:
def pixel(z0):
    # TODO: iterar con jax.lax.while_loop hasta converger o alcanzar MAX_ITER.
    # Debe retornar (root_idx, count) del pixel.
    pass


@jax.jit
def newton_jax(z0):
    # TODO: vectorizar pixel sobre la grid de dos dimensiones con jax.vmap.
    pass

z0_jax = jnp.asarray(z0)
root_jx, count_jx = newton_jax(z0_jax)
print("mismatch fraction:", mismatch_fraction(root_numpy, root_jx))
print(f"jax: {medir_jax(z0_jax):.4f} s")

## (d)

In [ ]:
Ws = [1024, 2048, 4096, 8192]
# TODO: medir el tiempo de las tres versiones para las tres regiones y para cada W en Ws.
# Mostrar las imágenes de las tres regiones con W = 1024 usando graficar,
# y presentar los tiempos en una tabla o gráfico.
# Medir también NumPy y JAX en un computador propio (sin GPU) para comparar.